# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guided approach for loading and exploring the FAIR² dataset using the `mlcroissant` library. All dataset elements are referenced by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant
# Jupyter-specific: silence possible restart warnings
%reload_ext autoreload

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This will initialize the Croissant dataset and print a summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Set Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset metadata and print a summary
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print("Description:")
print(metadata.description)

# Print dataset identifier and license for FAIR-ness
print(f"\nIdentifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview

Review available **record sets**, **fields**, and their `@id` using the Croissant metadata and `mlcroissant` API.
This helps you discover which data tables (record sets) and columns (fields) are available.

**Note**: All entities are referenced by their `@id` as per Croissant best practice.

In [ ]:
# List Record Sets and their fields using the Croissant schema

record_sets = list(dataset.record_sets)
print(f"{len(record_sets)} Record sets found:")
for rs in record_sets:
    print(f"  - @id: {rs.id} | name: {getattr(rs, 'name', 'NO NAME')}")
    print("    Fields:")
    for f in rs.fields:
        print(f"      - @id: {f.id} | name: {getattr(f, 'name', f.id)} | dataType: {getattr(f, 'data_type', 'unknown')}")
    print()

Let's preview the actual records from each record set using their Croissant `@id`.

*We keep the output size small for clarity; for full record lists, iterate without slicing.*

In [ ]:
# Preview the first few records from each record set (using @id)

for rs in record_sets:
    print(f"\nPreviewing records for Record Set: {rs.id}")
    iterator = dataset.records(record_set=rs.id)
    for i, row in enumerate(iterator):
        pprint.pprint(row)
        if i > 2:
            print("...")
            break

## 3. Data Extraction

Extract all available record sets into pandas DataFrames for flexible analysis. Each data table may have its own schema, so we load all using their `@id`.

Use the `@id` you found in the overview for precise data access.

In [ ]:
# Load all record sets into DataFrames, keying by record set @id

dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records into DataFrame for record set @id: {rs.id}")

# For demonstration, pick the first record set (as example – replace as needed)
example_record_set_id = record_sets[0].id

print(f"\nColumns in {example_record_set_id}:")
print(dataframes[example_record_set_id].columns.tolist())

print("\nFirst 5 rows:")
display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps using your preferred numeric or categorical fields referenced via Croissant `@id` (or matching DataFrame columns).

Example operations shown:
- Filtering records with numeric values above a threshold
- Normalizing numeric fields
- Grouping by a categorical field

Inspect the DataFrame columns above to choose suitable fields.

In [ ]:
# Example EDA: Replace field names according to your dataset schema
# We'll use the first DataFrame (example_record_set_id) for illustration.
df = dataframes[example_record_set_id]

# Identify candidate numeric and categorical fields
numeric_candidates = df.select_dtypes(include=["int", "float"]).columns.tolist()
print("Numeric fields found:", numeric_candidates)
if len(numeric_candidates) > 0:
    numeric_field = numeric_candidates[0]  # Pick first numeric for demonstration
else:
    print("No numeric fields found – please adjust selection.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 1
    print(f"\nFiltering {numeric_field} > {threshold}")
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records.")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nFirst 5 normalized {numeric_field} values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a candidate categorical field if present
    cat_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
    print("\nCategorical fields found:", cat_candidates)
    if len(cat_candidates) > 0:
        group_field = cat_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head(10))
else:
    print("No numeric field available for filtering and EDA.")

## 5. Visualization

Visualize numeric field distributions and, if available, relationships with categorical variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field (if present)
if numeric_field and len(filtered_df) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field], bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.show()

# Boxplot by categorical field (if present)
if numeric_field and 'group_field' in locals() and group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=25)
    plt.show()

## 6. Conclusion

In this notebook, you learned how to:
- Load a dataset described by a Croissant schema with `mlcroissant`
- Explore its metadata and discover the available record sets/fields by `@id`
- Extract records to DataFrames and analyze them
- Perform example filtering, transformation, and grouping
- Visualize numeric and categorical field relationships

**Next Steps:** Use this workflow with any dataset using a Croissant schema. For deeper domain analysis, consult the dataset's official schema and documentation using record set and field `@id`.

For reference, see: [FAIR² Dataset DOI:10.71728/senscience.qs2f-h81p](https://doi.org/10.71728/senscience.qs2f-h81p)
